# LawBot GRPO fine-tuning and evaluation

Optional Phase 4 experiment. This notebook creates base-model rejected-answer candidates for the 40-row preference seed CSV, then fine-tunes a 4-bit QLoRA policy with TRL GRPO using four small rule-based reward components. It starts from the Phase 2 SFT adapter when present and compares pre-GRPO against post-GRPO generation on a fixed 30-row validation sample.

**Important:** the training set is tiny and the rewards are weak proxies. They do not establish legal correctness. Inspect `reports/preference_candidates.csv` manually; generated candidates are marked unverified. Use retrieved sources for grounded answers and clearly label model-only legal answers as unverified. Educational use only, not legal advice.

Use Python 3.10+, a Colab/Kaggle T4 GPU, and the project's pinned `requirements-training.txt`. Run cells in order. No result values are prefilled.


In [ ]:
import os, sys, csv, json, gc, math, random, subprocess
from pathlib import Path
from collections import Counter
import numpy as np
import torch

ROOT_CANDIDATES = ([Path(os.environ["LAWBOT_PROJECT_DIR"]).expanduser()]
                   if os.environ.get("LAWBOT_PROJECT_DIR") else [])
ROOT_CANDIDATES += [Path.cwd(), Path.cwd().parent, Path("/content/lawbot"), Path("/kaggle/working/lawbot")]
PROJECT_DIR = next((p.resolve() for p in ROOT_CANDIDATES if (p / "src" / "reward.py").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Make lawbot/ available in the runtime or set LAWBOT_PROJECT_DIR.")
os.chdir(PROJECT_DIR); sys.path.insert(0, str(PROJECT_DIR))
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError("This QLoRA notebook requires a CUDA GPU (e.g. Colab/Kaggle T4).")
print("Project:", PROJECT_DIR, "GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Create 40 base-model rejected-answer candidates for manual inspection.
from unsloth import FastLanguageModel
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PREF_SEED = PROJECT_DIR / "reports" / "preference_seed.csv"
PREF_CANDIDATES = PROJECT_DIR / "reports" / "preference_candidates.csv"
SYSTEM_PROMPT = "You are LawBot, an assistant for Indian law. Cite sections/articles. Say you are unsure if you don't know."
pref_rows = list(csv.DictReader(PREF_SEED.open(encoding="utf-8-sig", newline="")))
if len(pref_rows) < 30 or len(pref_rows) > 50:
    raise ValueError(f"Expected 30-50 preference rows, found {len(pref_rows)}")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=1024, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(base_model)
for row in pref_rows:
    messages = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":row["prompt"]}]
    prompt = base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        generated = base_model.generate(**inputs, max_new_tokens=192, do_sample=True,
                                        temperature=0.8, top_p=0.9,
                                        pad_token_id=base_tokenizer.eos_token_id)
    row["rejected"] = base_tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    row["generation_status"] = "generated by base model; needs human review"
    row["human_verified"] = "no"
with PREF_CANDIDATES.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=pref_rows[0].keys()); writer.writeheader(); writer.writerows(pref_rows)
print(f"Wrote {len(pref_rows)} hand-reviewable pairs to {PREF_CANDIDATES}. Review chosen/rejected text; do not assume the dataset answer is always correct.")
del base_model, base_tokenizer; gc.collect(); torch.cuda.empty_cache()


## Prepare the GRPO prompts and rule-based rewards

GRPO uses the Q&A training references as scoring signals; it does not train directly on the preference CSV. The generated preference pairs are a separate review artifact as requested. Eight synthetic out-of-scope prompts teach the abstention reward.


In [ ]:
from src.reward import citation_match_reward, answer_similarity_reward, length_format_reward, abstention_reward

from datasets import Dataset
from transformers import AutoTokenizer
tokenizer_for_data = AutoTokenizer.from_pretrained(BASE_MODEL)

train_rows = [json.loads(line) for line in (PROJECT_DIR / "data/processed/train.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]

val_rows = [json.loads(line) for line in (PROJECT_DIR / "data/processed/val.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]

source_groups = {}

for row in train_rows:

    source_groups.setdefault(row["source"], []).append(row)

rng = random.Random(SEED)

for group in source_groups.values(): rng.shuffle(group)

selected = []

while len(selected) < 40:

    advanced = False

    for source in sorted(source_groups):

        if source_groups[source] and len(selected) < 40:

            selected.append(source_groups[source].pop()); advanced = True

    if not advanced: break

grpo_rows = []

for row in selected:

    messages = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":row["instruction"]}]

    prompt = tokenizer_for_data.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    grpo_rows.append({"prompt":prompt, "reference_answer":row["output"],

                      "references":row.get("references", []), "is_answerable":True, "source":row["source"]})

unanswerable = [

    "What is the weather in Delhi today?",

    "Who won today's cricket match and what was the score?",

    "What is the current price of Bitcoin in Indian rupees?",

    "What is the latest population count of Mumbai today?",

    "Which restaurant near me is open right now?",

    "What will the Supreme Court decide in a future case next month?",

    "What is the current temperature inside my home?",

    "Give the exact outcome of a case that has not yet been filed.",

]

for question in unanswerable:

    messages = [{"role":"system","content":SYSTEM_PROMPT + " If you lack reliable information, reply exactly: I don't have enough evidence in my sources to answer this."},

                {"role":"user","content":question}]

    prompt = tokenizer_for_data.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    grpo_rows.append({"prompt":prompt, "reference_answer":"", "references":[],

                      "is_answerable":False, "source":"synthetic_unanswerable"})

grpo_dataset = Dataset.from_list(grpo_rows).shuffle(seed=SEED)

print("GRPO prompts:", len(grpo_dataset), "sources:", Counter(r["source"] for r in grpo_rows))


## Load the SFT starting point and run GRPO

The config uses two generations per prompt, a single prompt per device batch and gradient accumulation to keep the effective generation group valid on one GPU. If memory runs out, reduce `max_completion_length` or set `per_device_train_batch_size=1` (already set) and gradient-checkpointing on (already enabled). Training is intentionally short; expect noisy updates.


In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import GRPOConfig, GRPOTrainer
ADAPTER_DIR = PROJECT_DIR / "adapters" / "lawbot-qwen1.5b"
GRPO_DIR = PROJECT_DIR / "adapters" / "lawbot-qwen1.5b-grpo"
START_MODEL = BASE_MODEL
policy, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=1024,
                                                       load_in_4bit=True, dtype=None)
if ADAPTER_DIR.exists() and (ADAPTER_DIR / "adapter_config.json").exists():
    policy = PeftModel.from_pretrained(policy, str(ADAPTER_DIR), is_trainable=True)
    print("Starting from Phase 2 SFT adapter.")
else:
    policy = FastLanguageModel.get_peft_model(policy, r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.0, bias="none", use_gradient_checkpointing="unsloth", random_state=SEED)
    print("Phase 2 adapter missing: starting a fresh QLoRA adapter from the base model.")
FastLanguageModel.for_training(policy)
policy.config.use_cache = False
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer_for_data = tokenizer
training_args = GRPOConfig(
    output_dir=str(GRPO_DIR), per_device_train_batch_size=1, gradient_accumulation_steps=2,
    num_generations=2, max_prompt_length=512, max_completion_length=160,
    learning_rate=5e-6, max_steps=12, beta=0.0, temperature=0.8,
    gradient_checkpointing=True, optim="paged_adamw_8bit", logging_steps=1,
    save_strategy="no", report_to="none", remove_unused_columns=False,
    seed=SEED, data_seed=SEED, use_vllm=False)
trainer = GRPOTrainer(model=policy,
    reward_funcs=[citation_match_reward, answer_similarity_reward, length_format_reward, abstention_reward],
    args=training_args, train_dataset=grpo_dataset, processing_class=tokenizer)
train_result = trainer.train()
trainer.save_model(str(GRPO_DIR)); tokenizer.save_pretrained(str(GRPO_DIR))
print("Saved GRPO adapter:", GRPO_DIR)


## Compare pre-GRPO and post-GRPO on validation

The notebook evaluates a fixed seeded sample of 30 held-out questions. It reports conditional validation loss/perplexity, ROUGE-L, BERTScore F1, and exact section/article citation-set accuracy, with source-wise ROUGE-L. This is a small noisy sample; score increases do not prove safer or legally correct behavior.


In [ ]:
# Generation and teacher-forced conditional loss metrics for pre/post adapters.
from src.evaluation import rouge_l_f1, citation_match
from unsloth import FastLanguageModel
import evaluate
eval_rng = random.Random(2026)
val_sample = val_rows.copy(); eval_rng.shuffle(val_sample); val_sample = val_sample[:30]
def run_eval(adapter_path, label):
    model_name = str(adapter_path) if adapter_path and (Path(adapter_path) / "adapter_config.json").exists() else BASE_MODEL
    ev_model, ev_tok = FastLanguageModel.from_pretrained(model_name=model_name, max_seq_length=1024,
                                                          load_in_4bit=True, dtype=None)
    FastLanguageModel.for_inference(ev_model)
    pred_texts, ref_texts, losses = [], [], []
    for row in val_sample:
        messages = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":row["instruction"]}]
        prompt = ev_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = ev_tok(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            generated = ev_model.generate(**inputs, max_new_tokens=192, do_sample=False,
                                          pad_token_id=ev_tok.eos_token_id)
        pred = ev_tok.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        pred_texts.append(pred); ref_texts.append(row["output"])
        # Mask the prompt; compute loss only on the reference completion.
        full_messages = messages + [{"role":"assistant","content":row["output"]}]
        prefix_ids = ev_tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        full_ids = ev_tok.apply_chat_template(full_messages, tokenize=True, add_generation_prompt=False)
        if isinstance(full_ids, dict): full_ids = full_ids["input_ids"]
        if isinstance(prefix_ids, dict): prefix_ids = prefix_ids["input_ids"]
        full_ids = torch.tensor([full_ids], device="cuda")
        labels = full_ids.clone(); labels[:, :len(prefix_ids)] = -100
        with torch.no_grad(): out = ev_model(input_ids=full_ids, labels=labels)
        if torch.isfinite(out.loss): losses.append(float(out.loss.detach().cpu()))
    exact = [citation_match(pred, row.get("references", [])) for pred,row in zip(pred_texts,val_sample)]
    eligible = [v for v in exact if v is not None]
    source_wise = {}
    for source in sorted({r["source"] for r in val_sample}):
        indices = [i for i, r in enumerate(val_sample) if r["source"] == source]
        source_wise[source] = float(np.mean([rouge_l_f1(pred_texts[i], ref_texts[i]) for i in indices]))
    result = {"model":label, "n":len(val_sample),
              "validation_loss":float(np.mean(losses)) if losses else None,
              "perplexity":float(math.exp(min(20,float(np.mean(losses))))) if losses else None,
              "rouge_l_f1":float(np.mean([rouge_l_f1(p,r) for p,r in zip(pred_texts,ref_texts)])),
              "citation_accuracy":float(np.mean(eligible)) if eligible else None,
              "citation_eligible_n":len(eligible), "source_wise_rouge_l":source_wise, "predictions":pred_texts}
    del ev_model, ev_tok; gc.collect(); torch.cuda.empty_cache()
    return result

pre_adapter = ADAPTER_DIR if (ADAPTER_DIR / "adapter_config.json").exists() else None
pre = run_eval(pre_adapter, "pre-GRPO (Phase 2 adapter if available; otherwise base)")
post = run_eval(GRPO_DIR, "post-GRPO")
bertscore = evaluate.load("bertscore")
for item in [pre, post]:
    item["bertscore_f1"] = float(np.mean(bertscore.compute(predictions=item["predictions"],
        references=[r["output"] for r in val_sample], lang="en", device="cpu", batch_size=4)["f1"]))
    item.pop("predictions", None)
# Source-wise ROUGE-L for both models is recorded in the report.


In [ ]:
# Write comparison tables and honest limitations without optional reporting packages.
def markdown_table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    out.extend("| " + " | ".join(str(value) for value in row) + " |" for row in rows)
    return "\n".join(out)
metric_keys = ["model","n","validation_loss","perplexity","rouge_l_f1","bertscore_f1","citation_accuracy","citation_eligible_n"]
table = markdown_table(metric_keys, [[item.get(key) for key in metric_keys] for item in [pre, post]])
source_names = sorted(set(pre["source_wise_rouge_l"]) | set(post["source_wise_rouge_l"]))
source_headers = ["source", "pre-GRPO ROUGE-L", "post-GRPO ROUGE-L"]
source_rows = [[source, pre["source_wise_rouge_l"].get(source), post["source_wise_rouge_l"].get(source)] for source in source_names]
source_table = markdown_table(source_headers, source_rows)
report_path = PROJECT_DIR / "reports" / "grpo_eval_results.md"
report_path.write_text("# Phase 4 GRPO evaluation\n\n" + table +
    "\n\n## Source-wise ROUGE-L\n\n" + source_table +
    "\n\n## Method and limitations\n\n"
    "Seeded 30-row sample from the held-out validation split. Pre-GRPO means the Phase 2 adapter when present; if absent, the base model. GRPO used 40 answerable training examples and 8 synthetic unanswerable prompts, four rule-based reward components, 4-bit QLoRA, and 12 optimizer steps. The reward signals use reference overlap and extracted citation strings; neither is a legal correctness judge.\n\n"
    "This tiny, synthetic-reward run is exploratory and may be noisy or overfit. BERTScore and citation metrics do not establish legal reliability. Use RAG evidence for grounding, verify current statutes, and do not use this as legal advice.\n", encoding="utf-8")
print(table); print("Wrote", report_path)
